In [1]:
# =========================================================
# 0. Install
# =========================================================
!pip -q install transformers accelerate


# =========================================================
# 1. Imports
# =========================================================
import copy
import json
import math
import random
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import StratifiedShuffleSplit

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, Subset
from tqdm.auto import tqdm

from transformers import ASTForAudioClassification


# =========================================================
# 2. Mount Drive and copy dataset to local disk
# =========================================================
from google.colab import drive
drive.mount("/content/drive")

DRIVE_ROOT = Path("/content/drive/MyDrive/kaggle_cs3780_sp26")
DRIVE_DATA_DIR = DRIVE_ROOT / "lowdim_export" / "reduced_64x32"
LOCAL_ROOT = Path("/content/local_data")
LOCAL_DATA_DIR = LOCAL_ROOT / "reduced_64x32"

if LOCAL_DATA_DIR.exists():
    print(f"Local dataset already exists at {LOCAL_DATA_DIR}")
else:
    LOCAL_ROOT.mkdir(parents=True, exist_ok=True)
    print(f"Copying dataset from {DRIVE_DATA_DIR} to {LOCAL_DATA_DIR} ...")
    shutil.copytree(DRIVE_DATA_DIR, LOCAL_DATA_DIR)
    print("Copy complete.")

print("Local train dir:", LOCAL_DATA_DIR / "train")
print("Local test dir:", LOCAL_DATA_DIR / "test")
print("Local solution:", LOCAL_DATA_DIR / "solution.csv")


# =========================================================
# 3. Config
# =========================================================
BASE_DATA_ROOT = LOCAL_ROOT
TRAIN_SUBDIR = Path("reduced_64x32/train")
TEST_SUBDIR = Path("reduced_64x32/test")
SOLUTION_CSV = LOCAL_DATA_DIR / "solution.csv"

OUTPUT_DIR = DRIVE_ROOT / "ast_progressive_llrd"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_SUMMARY_JSON = OUTPUT_DIR / "ast_progressive_llrd_metrics.json"
OUTPUT_ENSEMBLE_CSV = OUTPUT_DIR / "ast_progressive_llrd_submission.csv"

MODEL_NAME = "MIT/ast-finetuned-audioset-10-10-0.4593"

# AST-style spectrogram grid
AST_TIME_BINS = 1024
AST_MEL_BINS = 128

BATCH_SIZE = 12
EPOCHS = 18
WARMUP_EPOCHS = 1
NUM_WORKERS = 2
PATIENCE = 5

LABEL_SMOOTHING = 0.02
WEIGHT_DECAY = 1e-4
GRAD_CLIP_NORM = 1.0
USE_AMP = True
USE_TTA = True
EMA_DECAY = 0.999

# progressive unfreezing
HEAD_ONLY_EPOCHS = 2
LAST4_EPOCHS = 4   # epochs 3-4: unfreeze last 4 encoder layers
# epoch 5+ full model

# base learning rates
LR_HEAD = 2e-4
LR_LAST4 = 5e-5
LR_FULL_TOP = 2e-5
LR_FULL_BOTTOM = 2e-6   # earliest layers get the smallest LR

SEEDS = [2026, 2027, 2028]

# try True as an ablation if needed
INVERT_SPEC = False

# use a val split from train; keep test only for final reporting
VAL_SIZE = 0.15


# =========================================================
# 4. Utilities
# =========================================================
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
PIN_MEMORY = torch.cuda.is_available()

print("Using device:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


def save_json(obj, path: Path):
    with path.open("w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2)


def model_num_params(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters())


class ModelEMA:
    def __init__(self, model, decay=0.999):
        self.decay = decay
        self.ema = copy.deepcopy(model).eval()
        for p in self.ema.parameters():
            p.requires_grad = False

    @torch.no_grad()
    def update(self, model):
        msd = model.state_dict()
        for k, v in self.ema.state_dict().items():
            if v.dtype.is_floating_point:
                v.copy_(v * self.decay + msd[k].detach() * (1.0 - self.decay))
            else:
                v.copy_(msd[k])


def cosine_lr_lambda(current_epoch, total_epochs, warmup_epochs):
    if current_epoch < warmup_epochs:
        return float(current_epoch + 1) / float(max(1, warmup_epochs))
    progress = (current_epoch - warmup_epochs) / float(max(1, total_epochs - warmup_epochs))
    return 0.5 * (1.0 + math.cos(math.pi * progress))


# =========================================================
# 5. PNG spectrogram -> AST tensor
# =========================================================
def png_to_ast_input(
    img: Image.Image,
    time_bins: int = AST_TIME_BINS,
    mel_bins: int = AST_MEL_BINS,
    invert_spec: bool = INVERT_SPEC,
    train: bool = False,
) -> torch.Tensor:
    img = img.convert("L")
    img = img.resize((time_bins, mel_bins), resample=Image.BILINEAR)
    arr = np.array(img).astype(np.float32) / 255.0

    if invert_spec:
        arr = 1.0 - arr

    if train:
        # time shift
        if random.random() < 0.7:
            shift = random.randint(-16, 16)
            arr = np.roll(arr, shift=shift, axis=1)

        # freq shift
        if random.random() < 0.3:
            shift = random.randint(-4, 4)
            arr = np.roll(arr, shift=shift, axis=0)

        # time mask
        if random.random() < 0.6:
            t = random.randint(16, 96)
            t0 = random.randint(0, max(0, time_bins - t))
            arr[:, t0:t0+t] = 0.0

        # freq mask
        if random.random() < 0.6:
            f = random.randint(6, 24)
            f0 = random.randint(0, max(0, mel_bins - f))
            arr[f0:f0+f, :] = 0.0

        # contrast / brightness
        if random.random() < 0.5:
            scale = random.uniform(0.85, 1.2)
            bias = random.uniform(-0.05, 0.05)
            arr = np.clip(arr * scale + bias, 0.0, 1.0)

    mean = arr.mean()
    std = arr.std()
    arr = (arr - mean) / (std + 1e-6)

    # AST expects (time, mel)
    arr = arr.T
    return torch.tensor(arr, dtype=torch.float32)


# =========================================================
# 6. Dataset
# =========================================================
class TrainFolderASTDataset(Dataset):
    def __init__(self, root_dir: Path, train: bool = False):
        self.root_dir = root_dir
        self.train = train

        self.classes = sorted([p.name for p in root_dir.iterdir() if p.is_dir()])
        if not self.classes:
            raise FileNotFoundError(f"No class folders found under {root_dir}")

        self.class_to_idx = {cls_name: i for i, cls_name in enumerate(self.classes)}
        self.idx_to_class = {i: cls_name for cls_name, i in self.class_to_idx.items()}

        self.samples = []
        for cls_name in self.classes:
            class_dir = root_dir / cls_name
            for img_path in sorted(class_dir.rglob("*.png")):
                self.samples.append((img_path, self.class_to_idx[cls_name]))

        if not self.samples:
            raise FileNotFoundError(f"No PNG files found under {root_dir}")

        self.labels = [label for _, label in self.samples]

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        with Image.open(img_path) as img:
            x = png_to_ast_input(img, train=self.train)
        return x, label


class TestWithSolutionASTDataset(Dataset):
    def __init__(self, test_dir: Path, solution_csv: Path, class_to_idx: dict[str, int]):
        self.test_dir = test_dir
        self.class_to_idx = class_to_idx

        solution_df = pd.read_csv(solution_csv)
        if "file_name" not in solution_df.columns or "label" not in solution_df.columns:
            raise ValueError("solution.csv must contain columns: file_name, label")

        solution_map = dict(zip(solution_df["file_name"], solution_df["label"]))

        self.samples = []
        image_paths = sorted(test_dir.rglob("*.png"))
        if not image_paths:
            raise FileNotFoundError(f"No PNG files found under {test_dir}")

        for img_path in image_paths:
            fname = img_path.name
            if fname not in solution_map:
                raise KeyError(f"{fname} not found in solution.csv")
            label_name = solution_map[fname]
            if label_name not in class_to_idx:
                raise KeyError(f"Label {label_name} from solution.csv not found in training classes")
            label_idx = class_to_idx[label_name]
            self.samples.append((img_path, label_idx))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        with Image.open(img_path) as img:
            x = png_to_ast_input(img, train=False)
        return x, label, img_path.name


class TestUnlabeledASTDataset(Dataset):
    def __init__(self, test_dir: Path):
        self.test_dir = test_dir
        self.image_paths = sorted(test_dir.rglob("*.png"))
        if not self.image_paths:
            raise FileNotFoundError(f"No PNG files found under {test_dir}")

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        with Image.open(img_path) as img:
            x = png_to_ast_input(img, train=False)
        return x, img_path.name


# =========================================================
# 7. Load data
# =========================================================
train_dir = BASE_DATA_ROOT / TRAIN_SUBDIR
test_dir = BASE_DATA_ROOT / TEST_SUBDIR

full_train_dataset = TrainFolderASTDataset(train_dir, train=False)

sss = StratifiedShuffleSplit(n_splits=1, test_size=VAL_SIZE, random_state=42)
train_indices, val_indices = next(sss.split(np.zeros(len(full_train_dataset)), full_train_dataset.labels))

train_dataset = Subset(TrainFolderASTDataset(train_dir, train=True), train_indices)
val_dataset = Subset(TrainFolderASTDataset(train_dir, train=False), val_indices)

test_eval_dataset = TestWithSolutionASTDataset(
    test_dir=test_dir,
    solution_csv=SOLUTION_CSV,
    class_to_idx=full_train_dataset.class_to_idx,
)
test_pred_dataset = TestUnlabeledASTDataset(test_dir=test_dir)

persistent = NUM_WORKERS > 0

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    persistent_workers=persistent,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    persistent_workers=persistent,
)

test_eval_loader = DataLoader(
    test_eval_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    persistent_workers=persistent,
)

test_pred_loader = DataLoader(
    test_pred_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    persistent_workers=persistent,
)

num_classes = len(full_train_dataset.classes)
idx_to_class = full_train_dataset.idx_to_class

print("num_classes =", num_classes)
print("num_train =", len(full_train_dataset))
print("num_val =", len(val_dataset))
print("num_test =", len(test_eval_dataset))


# =========================================================
# 8. Model
# =========================================================
def build_model(num_classes: int):
    model = ASTForAudioClassification.from_pretrained(
        MODEL_NAME,
        num_labels=num_classes,
        ignore_mismatched_sizes=True,
    )
    return model


def set_requires_grad(module, flag: bool):
    for p in module.parameters():
        p.requires_grad = flag


def configure_stage(model: nn.Module, stage: str):
    """
    stage:
      - "head"
      - "last4"
      - "full"
    """
    # freeze everything first
    for p in model.parameters():
        p.requires_grad = False

    # classifier always trainable
    set_requires_grad(model.classifier, True)

    # AST backbone in HF: model.audio_spectrogram_transformer.encoder.layer
    encoder_layers = model.audio_spectrogram_transformer.encoder.layer

    if stage == "head":
        pass

    elif stage == "last4":
        # unfreeze final 4 encoder layers + layernorm if present
        for layer in encoder_layers[-4:]:
            set_requires_grad(layer, True)
        if hasattr(model.audio_spectrogram_transformer, "layernorm"):
            set_requires_grad(model.audio_spectrogram_transformer.layernorm, True)

    elif stage == "full":
        for p in model.parameters():
            p.requires_grad = True

    else:
        raise ValueError(f"Unknown stage: {stage}")


def make_optimizer_for_stage(model: nn.Module, stage: str):
    """
    layer-wise LR decay for AST
    """
    if stage == "head":
        params = [
            {
                "params": [p for p in model.classifier.parameters() if p.requires_grad],
                "lr": LR_HEAD,
                "weight_decay": WEIGHT_DECAY,
            }
        ]
        return torch.optim.AdamW(params)

    encoder_layers = model.audio_spectrogram_transformer.encoder.layer
    param_groups = []

    # classifier
    param_groups.append({
        "params": [p for p in model.classifier.parameters() if p.requires_grad],
        "lr": LR_LAST4 if stage == "last4" else LR_FULL_TOP,
        "weight_decay": WEIGHT_DECAY,
    })

    if stage == "last4":
        for layer in encoder_layers[-4:]:
            ps = [p for p in layer.parameters() if p.requires_grad]
            if ps:
                param_groups.append({
                    "params": ps,
                    "lr": LR_LAST4,
                    "weight_decay": WEIGHT_DECAY,
                })

        if hasattr(model.audio_spectrogram_transformer, "layernorm"):
            ps = [p for p in model.audio_spectrogram_transformer.layernorm.parameters() if p.requires_grad]
            if ps:
                param_groups.append({
                    "params": ps,
                    "lr": LR_LAST4,
                    "weight_decay": WEIGHT_DECAY,
                })

        return torch.optim.AdamW(param_groups)

    # full model: layer-wise lr decay
    num_layers = len(encoder_layers)
    decay_rate = (LR_FULL_BOTTOM / LR_FULL_TOP) ** (1 / max(num_layers - 1, 1))

    # embeddings / patch projection / pos embeddings
    if hasattr(model.audio_spectrogram_transformer, "embeddings"):
        emb_params = [p for p in model.audio_spectrogram_transformer.embeddings.parameters() if p.requires_grad]
        if emb_params:
            param_groups.append({
                "params": emb_params,
                "lr": LR_FULL_BOTTOM,
                "weight_decay": WEIGHT_DECAY,
            })

    # encoder layers: early small lr, late big lr
    for i, layer in enumerate(encoder_layers):
        lr_i = LR_FULL_BOTTOM * (decay_rate ** i)
        ps = [p for p in layer.parameters() if p.requires_grad]
        if ps:
            param_groups.append({
                "params": ps,
                "lr": lr_i,
                "weight_decay": WEIGHT_DECAY,
            })

    if hasattr(model.audio_spectrogram_transformer, "layernorm"):
        ps = [p for p in model.audio_spectrogram_transformer.layernorm.parameters() if p.requires_grad]
        if ps:
            param_groups.append({
                "params": ps,
                "lr": LR_FULL_TOP,
                "weight_decay": WEIGHT_DECAY,
            })

    return torch.optim.AdamW(param_groups)


# =========================================================
# 9. Train / Eval / Predict
# =========================================================
@torch.no_grad()
def run_eval_epoch(eval_model, loader, criterion, device, use_amp=True, use_tta=False):
    eval_model.eval()
    total_loss = 0.0
    all_preds = []
    all_labels = []
    all_file_names = []

    use_amp = use_amp and device.type == "cuda"

    for batch in tqdm(loader, leave=False):
        if len(batch) == 3:
            input_values, labels, file_names = batch
            all_file_names.extend(file_names)
        else:
            input_values, labels = batch

        input_values = input_values.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        if use_tta:
            variants = [
                input_values,
                torch.roll(input_values, shifts=8, dims=1),
                torch.roll(input_values, shifts=-8, dims=1),
                torch.roll(input_values, shifts=2, dims=2),
                torch.roll(input_values, shifts=-2, dims=2),
            ]
            logits_sum = 0.0
            for v in variants:
                with torch.autocast(device_type="cuda", enabled=use_amp):
                    logits_sum = logits_sum + eval_model(input_values=v).logits
            logits = logits_sum / len(variants)
        else:
            with torch.autocast(device_type="cuda", enabled=use_amp):
                logits = eval_model(input_values=input_values).logits

        loss = criterion(logits, labels)
        total_loss += loss.item() * input_values.size(0)
        preds = logits.argmax(dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / len(loader.dataset)
    acc = accuracy_score(all_labels, all_preds)
    return avg_loss, acc, np.array(all_labels), np.array(all_preds), all_file_names


def run_train_epoch(model, ema_model, loader, criterion, optimizer, scaler, device, use_amp=True):
    model.train()
    total_loss = 0.0
    all_preds = []
    all_labels = []

    use_amp = use_amp and device.type == "cuda"

    for input_values, labels in tqdm(loader, leave=False):
        input_values = input_values.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.autocast(device_type="cuda", enabled=use_amp):
            logits = model(input_values=input_values).logits
            loss = criterion(logits, labels)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
        scaler.step(optimizer)
        scaler.update()

        ema_model.update(model)

        total_loss += loss.item() * input_values.size(0)
        preds = logits.argmax(dim=1)

        all_preds.extend(preds.detach().cpu().numpy())
        all_labels.extend(labels.detach().cpu().numpy())

    avg_loss = total_loss / len(loader.dataset)
    acc = accuracy_score(all_labels, all_preds)
    return avg_loss, acc


@torch.no_grad()
def predict_logits(eval_model, loader, device, use_amp=True, use_tta=False):
    eval_model.eval()
    all_logits = []
    all_file_names = []
    use_amp = use_amp and device.type == "cuda"

    for input_values, file_names in tqdm(loader, leave=False):
        input_values = input_values.to(device, non_blocking=True)

        if use_tta:
            variants = [
                input_values,
                torch.roll(input_values, shifts=8, dims=1),
                torch.roll(input_values, shifts=-8, dims=1),
                torch.roll(input_values, shifts=2, dims=2),
                torch.roll(input_values, shifts=-2, dims=2),
            ]
            logits_sum = 0.0
            for v in variants:
                with torch.autocast(device_type="cuda", enabled=use_amp):
                    logits_sum = logits_sum + eval_model(input_values=v).logits
            logits = logits_sum / len(variants)
        else:
            with torch.autocast(device_type="cuda", enabled=use_amp):
                logits = eval_model(input_values=input_values).logits

        all_logits.append(logits.float().cpu())
        all_file_names.extend(list(file_names))

    all_logits = torch.cat(all_logits, dim=0).numpy()
    return all_logits, all_file_names


def logits_to_pred_df(logits: np.ndarray, file_names: list[str], idx_to_class: dict[int, str]):
    preds = logits.argmax(axis=1)
    return pd.DataFrame({
        "file_name": file_names,
        "label": [idx_to_class[int(i)] for i in preds],
    })


# =========================================================
# 10. One seed training
# =========================================================
def train_one_seed(seed: int):
    print("\n" + "=" * 80)
    print(f"Starting seed {seed}")
    set_seed(seed)

    seed_output_dir = OUTPUT_DIR / f"seed_{seed}"
    seed_output_dir.mkdir(parents=True, exist_ok=True)

    model = build_model(num_classes).to(DEVICE)
    ema_model = ModelEMA(model, decay=EMA_DECAY)
    criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
    scaler = torch.amp.GradScaler("cuda", enabled=(USE_AMP and DEVICE.type == "cuda"))

    current_stage = None
    optimizer = None
    scheduler = None

    history = []
    best_val_acc = -1.0
    best_epoch = -1
    best_stage = None
    best_state_dict = None
    best_ema_state_dict = None
    best_val_y_true = None
    best_val_y_pred = None
    epochs_without_improve = 0

    for epoch in range(1, EPOCHS + 1):
        if epoch <= HEAD_ONLY_EPOCHS:
            stage = "head"
        elif epoch <= LAST4_EPOCHS:
            stage = "last4"
        else:
            stage = "full"

        if stage != current_stage:
            print(f"[seed {seed}] switching to stage = {stage}")
            current_stage = stage
            configure_stage(model, stage)
            optimizer = make_optimizer_for_stage(model, stage)
            scheduler = torch.optim.lr_scheduler.LambdaLR(
                optimizer,
                lr_lambda=lambda ep: cosine_lr_lambda(ep, max(EPOCHS - epoch + 1, 1), WARMUP_EPOCHS)
            )

        print(f"\n[seed {seed}] Epoch {epoch}/{EPOCHS} | stage={stage}")

        train_loss, train_acc = run_train_epoch(
            model, ema_model, train_loader, criterion, optimizer, scaler, DEVICE, USE_AMP
        )

        val_loss, val_acc, y_true, y_pred, _ = run_eval_epoch(
            ema_model.ema, val_loader, criterion, DEVICE, USE_AMP, use_tta=USE_TTA
        )

        max_lr = max(pg["lr"] for pg in optimizer.param_groups)

        print(f"[seed {seed}] train_loss={train_loss:.4f}, train_acc={train_acc:.4f}")
        print(f"[seed {seed}] val_loss={val_loss:.4f}, val_acc={val_acc:.4f}, max_lr={max_lr:.6g}")

        history.append({
            "epoch": epoch,
            "stage": stage,
            "train_loss": float(train_loss),
            "train_accuracy": float(train_acc),
            "val_loss": float(val_loss),
            "val_accuracy": float(val_acc),
            "max_lr": float(max_lr),
        })

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_epoch = epoch
            best_stage = stage
            best_state_dict = copy.deepcopy(model.state_dict())
            best_ema_state_dict = copy.deepcopy(ema_model.ema.state_dict())
            best_val_y_true = y_true.copy()
            best_val_y_pred = y_pred.copy()
            epochs_without_improve = 0

            best_logits, best_file_names = predict_logits(
                ema_model.ema,
                test_pred_loader,
                DEVICE,
                use_amp=USE_AMP,
                use_tta=USE_TTA,
            )
            best_pred_df = logits_to_pred_df(best_logits, best_file_names, idx_to_class)
            best_pred_df.to_csv(seed_output_dir / "best_test_predictions.csv", index=False)
            np.save(seed_output_dir / "best_test_logits.npy", best_logits)
            print(f"[seed {seed}] Saved new best logits/CSV at epoch {epoch}")
        else:
            epochs_without_improve += 1

        scheduler.step()

        if epochs_without_improve >= PATIENCE:
            print(f"[seed {seed}] Early stopping after {PATIENCE} epochs without improvement.")
            break

    ema_model.ema.load_state_dict(best_ema_state_dict)

    test_loss, test_acc, _, _, _ = run_eval_epoch(
        ema_model.ema, test_eval_loader, criterion, DEVICE, USE_AMP, use_tta=USE_TTA
    )

    report = classification_report(
        best_val_y_true,
        best_val_y_pred,
        target_names=full_train_dataset.classes,
        output_dict=True,
        zero_division=0,
    )

    metrics = {
        "seed": seed,
        "config": {
            "model_name": MODEL_NAME,
            "ast_time_bins": AST_TIME_BINS,
            "ast_mel_bins": AST_MEL_BINS,
            "batch_size": BATCH_SIZE,
            "epochs": EPOCHS,
            "label_smoothing": LABEL_SMOOTHING,
            "weight_decay": WEIGHT_DECAY,
            "head_only_epochs": HEAD_ONLY_EPOCHS,
            "last4_epochs": LAST4_EPOCHS,
            "lr_head": LR_HEAD,
            "lr_last4": LR_LAST4,
            "lr_full_top": LR_FULL_TOP,
            "lr_full_bottom": LR_FULL_BOTTOM,
            "use_tta": USE_TTA,
            "invert_spec": INVERT_SPEC,
            "ema_decay": EMA_DECAY,
        },
        "num_train_total": len(full_train_dataset),
        "num_val": len(val_dataset),
        "num_test": len(test_eval_dataset),
        "num_classes": num_classes,
        "classes": full_train_dataset.classes,
        "model_num_params": model_num_params(model),
        "best_val_accuracy": float(best_val_acc),
        "best_epoch": int(best_epoch),
        "best_stage": best_stage,
        "test_accuracy": float(test_acc),
        "history": history,
        "classification_report": report,
    }

    save_json(metrics, seed_output_dir / "metrics.json")
    torch.save({
        "model_state_dict": best_state_dict,
        "ema_state_dict": best_ema_state_dict,
        "classes": full_train_dataset.classes,
        "best_epoch": best_epoch,
        "best_stage": best_stage,
        "best_val_accuracy": best_val_acc,
        "seed": seed,
    }, seed_output_dir / "best_model.pt")

    return {
        "seed": seed,
        "best_val_accuracy": float(best_val_acc),
        "best_epoch": int(best_epoch),
        "best_stage": best_stage,
        "test_accuracy": float(test_acc),
        "logits_path": str(seed_output_dir / "best_test_logits.npy"),
        "csv_path": str(seed_output_dir / "best_test_predictions.csv"),
    }


# =========================================================
# 11. Train all 3 seeds
# =========================================================
seed_summaries = []
all_seed_logits = []
reference_file_names = None

for seed in SEEDS:
    summary = train_one_seed(seed)
    seed_summaries.append(summary)

    logits = np.load(summary["logits_path"])
    all_seed_logits.append(logits)

    pred_df = pd.read_csv(summary["csv_path"])
    file_names = pred_df["file_name"].tolist()

    if reference_file_names is None:
        reference_file_names = file_names
    else:
        if reference_file_names != file_names:
            raise ValueError("File order mismatch across seeds. Cannot ensemble safely.")


# =========================================================
# 12. Ensemble logits and save final submission
# =========================================================
ensemble_logits = np.mean(np.stack(all_seed_logits, axis=0), axis=0)
ensemble_pred_df = logits_to_pred_df(ensemble_logits, reference_file_names, idx_to_class)
ensemble_pred_df.to_csv(OUTPUT_ENSEMBLE_CSV, index=False)

solution_df = pd.read_csv(SOLUTION_CSV)
solution_map = dict(zip(solution_df["file_name"], solution_df["label"]))
y_true_labels = [solution_map[f] for f in reference_file_names]
y_pred_labels = ensemble_pred_df["label"].tolist()
ensemble_test_acc = float(np.mean(np.array(y_true_labels) == np.array(y_pred_labels)))

summary = {
    "seeds": SEEDS,
    "seed_summaries": seed_summaries,
    "ensemble_test_accuracy": ensemble_test_acc,
    "ensemble_csv": str(OUTPUT_ENSEMBLE_CSV),
}

save_json(summary, OUTPUT_SUMMARY_JSON)

print("\n==== Final Ensemble Results ====")
for row in seed_summaries:
    print(
        f"seed={row['seed']} "
        f"val={row['best_val_accuracy']:.4f} "
        f"test={row['test_accuracy']:.4f} "
        f"best_epoch={row['best_epoch']} "
        f"stage={row['best_stage']}"
    )

print(f"ensemble_test_accuracy={ensemble_test_acc:.4f}")
print(f"Saved ensemble metrics to {OUTPUT_SUMMARY_JSON}")
print(f"Saved final ensemble submission to {OUTPUT_ENSEMBLE_CSV}")

ensemble_pred_df.head()

Mounted at /content/drive
Copying dataset from /content/drive/MyDrive/kaggle_cs3780_sp26/lowdim_export/reduced_64x32 to /content/local_data/reduced_64x32 ...
Copy complete.
Local train dir: /content/local_data/reduced_64x32/train
Local test dir: /content/local_data/reduced_64x32/test
Local solution: /content/local_data/reduced_64x32/solution.csv
Using device: cuda
GPU: NVIDIA A100-SXM4-40GB
num_classes = 9
num_train = 21898
num_val = 3285
num_test = 5454

Starting seed 2026


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/203 [00:00<?, ?it/s]

...

  0%|          | 0/455 [00:00<?, ?it/s]

[seed 2028] Saved new best logits/CSV at epoch 18


  0%|          | 0/455 [00:00<?, ?it/s]


==== Final Ensemble Results ====
seed=2026 val=0.4712 test=0.4762 best_epoch=18 stage=full
seed=2027 val=0.4728 test=0.4721 best_epoch=18 stage=full
seed=2028 val=0.4654 test=0.4741 best_epoch=18 stage=full
ensemble_test_accuracy=0.4800
Saved ensemble metrics to /content/drive/MyDrive/kaggle_cs3780_sp26/ast_progressive_llrd/ast_progressive_llrd_metrics.json
Saved final ensemble submission to /content/drive/MyDrive/kaggle_cs3780_sp26/ast_progressive_llrd/ast_progressive_llrd_submission.csv


,file_name,label
0,1.png,Nocturnal bird
1,10.png,Parrot
2,100.png,Other non-passerine bird
3,1000.png,Nocturnal bird
4,1001.png,Flycatcher
